In [10]:
# Importing Libs
!pip install pandera
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
import pandera as pa
from pandera import Column, DataFrameSchema

In [11]:
# Load Dataset
df = pd.read_excel('/content/Dataset for Data Analytics.xlsx')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [13]:
df.shape

(1200, 14)

In [14]:
# --- STAGE 1: INPUT (Securing Fidelity) ---
def secure_fidelity(data):
    # Create a copy to avoid SettingWithCopyWarning if data is a view
    data_copy = data.copy()

    # Rule: Drop rows < 5%, Impute 5-20%, KNN > 20%
    for col in data_copy.columns:
        missing_pct = data_copy[col].isnull().mean()

        if missing_pct == 0:
            continue # No missing values, skip

        if 0 < missing_pct < 0.05:
            # For small missing percentages, drop rows
            data_copy = data_copy.dropna(subset=[col])
        elif 0.05 <= missing_pct <= 0.20:
            # For moderate missing percentages
            if data_copy[col].dtype in ['float64', 'int64']:
                data_copy[col] = data_copy[col].fillna(data_copy[col].median())
            else: # Categorical or other types
                data_copy[col] = data_copy[col].fillna(data_copy[col].mode()[0])
        elif missing_pct > 0.20:
            # For high missing percentages
            if data_copy[col].dtype in ['float64', 'int64']:
                imputer = KNNImputer(n_neighbors=5)
                # KNNImputer expects a 2D array, so we pass data_copy[[col]]
                data_copy[col] = imputer.fit_transform(data_copy[[col]])[:, 0] # Extract the single column
            else: # Categorical or other types
                data_copy[col] = data_copy[col].fillna(data_copy[col].mode()[0])
    return data_copy

In [15]:
def neutralize_outliers(data):
    # Rule: Neutralize outliers using IQR and numpy.clip() to preserve row count
    for col in data.select_dtypes(include=[np.number]).columns:
        Q1, Q3 = data[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        data[col] = np.clip(data[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return data

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# --- STAGE 2: PROCESS (The Engine) ---
def process_engine(data):
    # Rule: Avoid loops, use Vectorized math

    # Identify numerical and categorical columns
    numerical_cols = data.select_dtypes(include=np.number).columns
    categorical_cols = data.select_dtypes(include=['object']).columns

    # Initialize a list to hold processed parts of the DataFrame
    processed_parts = []

    # Add numerical columns directly
    if not numerical_cols.empty:
        processed_parts.append(data[numerical_cols])

    # Handle one-hot encoding for categorical features
    if not categorical_cols.empty:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_data = encoder.fit_transform(data[categorical_cols])

        # Create a DataFrame from the encoded data with appropriate column names and index
        encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols), index=data.index)
        processed_parts.append(encoded_df)

    # Concatenate all processed parts
    if processed_parts:
        df_processed = pd.concat(processed_parts, axis=1)
    else:
        df_processed = pd.DataFrame(index=data.index) # Return an empty DataFrame if no columns

    return df_processed

In [17]:
def remove_collinear_features(data, threshold=0.80, protected_cols=None):
    # Rule: Collinearity Eradication
    if protected_cols is None:
        protected_cols = []

    # Ensure data is a DataFrame to use .select_dtypes
    if not isinstance(data, pd.DataFrame):
        return data

    numeric_data = data.select_dtypes(include=np.number)

    # Filter protected columns to only include those present in the numeric data
    protected_numeric_cols = [col for col in protected_cols if col in numeric_data.columns]

    # Columns to consider for collinearity removal (excluding protected ones)
    cols_to_consider = [col for col in numeric_data.columns if col not in protected_numeric_cols]

    if not cols_to_consider: # If no columns left to consider for dropping
        return data # No columns to drop based on collinearity

    corr_matrix = numeric_data[cols_to_consider].corr().abs()

    # Select upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features with correlation greater than threshold
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

    # Drop identified features from the original DataFrame (including non-numeric columns)
    df_reduced = data.drop(columns=to_drop, errors='ignore')
    return df_reduced

In [18]:
# --- STAGE 3: OUTPUT (Contracts & Serving) ---
# Rule: Runtime Contract Assertion
schema = DataFrameSchema(
    {
        "Quantity": Column(int, checks=pa.Check.gt(0)),
        "TotalPrice": Column(float, checks=pa.Check.gt(0.0)),
        # Add more specific validation for one-hot encoded columns or other expected features if needed.
        # Example: Column("Product_Monitor", float, checks=pa.Check.isin([0.0, 1.0])),
    },
    # Set strict=False to allow other columns (like one-hot encoded ones) to exist without explicit validation.
    # Set strict=True if you want to enforce that only explicitly defined columns are present.
    strict=False
)

df_clean = secure_fidelity(df)
df_processed = process_engine(df_clean)
df_processed = neutralize_outliers(df_processed)

# Define columns to protect from collinearity removal (e.g., those expected by schema)
protected_cols_for_collinearity = ["Quantity", "TotalPrice"]
df_processed = remove_collinear_features(df_processed, protected_cols=protected_cols_for_collinearity)

validated_data = schema.validate(df_processed)

print("Data successfully validated!")
display(validated_data.head())

Data successfully validated!


,Quantity,UnitPrice,ItemsInCart,TotalPrice,OrderID_ORD200000,OrderID_ORD200001,OrderID_ORD200002,OrderID_ORD200003,OrderID_ORD200004,OrderID_ORD200005,...,TrackingNumber_TRK99938245,TrackingNumber_TRK99985340,CouponCode_FREESHIP,CouponCode_SAVE10,CouponCode_WINTER15,ReferralSource_Email,ReferralSource_Facebook,ReferralSource_Google,ReferralSource_Instagram,ReferralSource_Referral
0,5,570.62,7,2853.10,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,151.35,3,302.70,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5,550.68,8,2753.40,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,273.19,5,273.19,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,626.01,8,2504.04,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
